# F01_A: NSD ROI Mega-Matrix MDS

Minimal reproduction notebook for manuscript Fig. 1A. This starts from the saved whole-brain `mega_matrix` pickle copied into `streamlined/data/processed/mega_mega_matrix.data`, runs MDS on `1 - mega_matrix`, and exports the plotted coordinates.
(see https://github.com/dawnfinzi/streams/blob/master/notebooks/Whole_brain_mega_matrix.ipynb)

Important: the original notebook used `MDS(dissimilarity="precomputed")` without `random_state`, so its exact coordinates were not deterministic. This notebook uses `random_state = 0` for a reproducible source-data export.

In [ ]:
from pathlib import Path
import pickle
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.manifold import MDS
from sklearn.metrics import euclidean_distances

from spacestream.core.paths import DATA_PATH

MATRIX_PATH = Path(DATA_PATH) / "processed" / "mega_mega_matrix.data"
DAWN = Path(DATA_PATH).resolve().parents[1]
SOURCE_DATA_DIR = DAWN / "manuscript_figure_source_data" / "data" / "main_text"
SOURCE_DATA_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 0
print(MATRIX_PATH)

## Load Mega Matrix

The pickle stores a one-element list containing the 112 x 112 ROI/subject/hemisphere correlation matrix.

In [ ]:
with MATRIX_PATH.open("rb") as f:
    loaded = pickle.load(f)

mega_matrix = loaded[0] if isinstance(loaded, (list, tuple)) else loaded
mega_matrix = np.asarray(mega_matrix, dtype=float)

assert mega_matrix.shape == (112, 112), mega_matrix.shape
assert np.allclose(mega_matrix, mega_matrix.T, equal_nan=True)

print("shape:", mega_matrix.shape)
print("min/max:", np.nanmin(mega_matrix), np.nanmax(mega_matrix))

## Run MDS

This matches the original transform, `mega_MDS.fit_transform(1 - mega_matrix)`, but fixes `random_state` for reproducibility.

In [ ]:
dissimilarity = 1 - mega_matrix

mega_MDS = MDS(
    n_components=2,
    dissimilarity="precomputed",
    random_state=RANDOM_STATE,
    n_init=20
)
embedding = mega_MDS.fit_transform(dissimilarity)
x, y = embedding.T

print("sklearn stress:", mega_MDS.stress_)
DE = euclidean_distances(embedding)
stress = 0.5 * np.sum((DE - dissimilarity) ** 2)
stress1 = np.sqrt(stress / (0.5 * np.sum(dissimilarity ** 2)))
print("manual stress:", stress)
print("Kruskal stress:", stress1)

## Build Source-Data Table

Rows follow the original matrix ordering: all left-hemisphere ROI-by-subject rows, then all right-hemisphere rows. ROI labels preserve the historical `Parietal` label and add `stream_display = Dorsal` for manuscript-facing naming.

In [ ]:
subjects = [f"{i:02d}" for i in range(1, 9)]
hemis = ["lh", "rh"]
roi_names = ["Early", "Midventral", "Midlateral", "Midparietal", "Ventral", "Lateral", "Parietal"]
stream_display = {"Parietal": "Dorsal"}

rows = []
idx = 0
for hemi in hemis:
    for roi_index, roi in enumerate(roi_names, start=1):
        for subject in subjects:
            rows.append({
                "matrix_index": idx,
                "subject": subject,
                "hemi": hemi,
                "roi_index": roi_index,
                "roi_raw": roi,
                "roi_display": stream_display.get(roi, roi),
                "mds_dim1": x[idx],
                "mds_dim2": y[idx],
                "random_state": RANDOM_STATE,
                "sklearn_stress": mega_MDS.stress_,
                "kruskal_stress": stress1,
            })
            idx += 1

coords = pd.DataFrame(rows)
coords.head()

In [ ]:
coords_file = SOURCE_DATA_DIR / "fig1a_mds_coordinates_random_state0.csv"
matrix_file = SOURCE_DATA_DIR / "fig1a_mega_matrix.csv"

coords.to_csv(coords_file, index=False)
pd.DataFrame(mega_matrix).to_csv(matrix_file, index=False, header=False)

print(coords_file)
print(matrix_file)

## Quick Plot Check

In [ ]:
roi_colors = {
    "Early": "#8f8f8f",
    "Midventral": "#e895bd",
    "Midlateral": "#9fb8ff",
    "Midparietal": "#7ee39a",
    "Ventral": "#DC267F",
    "Lateral": "#053bc2",
    "Parietal": "#006600",
}
markers = {"lh": "v", "rh": "o"}

fig, ax = plt.subplots(figsize=(7, 7))
for (roi, hemi), group in coords.groupby(["roi_raw", "hemi"]):
    ax.scatter(
        group["mds_dim1"],
        group["mds_dim2"],
        s=10 + 25 * group["subject"].astype(int),
        c=roi_colors[roi],
        marker=markers[hemi],
        edgecolors="white",
        linewidths=0.5,
        alpha=0.9,
        label=f"{roi}, {hemi}",
    )

ax.set_xlabel("MDS dimension 1")
ax.set_ylabel("MDS dimension 2")
ax.set_title("Fig 1A MDS from saved mega matrix")
ax.spines[["top", "right"]].set_visible(False)
plt.show()